# Evaluation — HARNN Word2Vec + Global

Load checkpoint đã train, vẽ toàn bộ biểu đồ đánh giá.

| Cell | Biểu đồ |
|------|----------|
| 1 | Load model + test set |
| 2 | Confusion Matrix L1 |
| 3 | F1 theo từng nhãn L2 |
| 4 | Precision-Recall curve |
| 5 | Phân phối xác suất dự đoán |


## Cell 1 — Load model & test set


In [ ]:
import json, numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score,
    precision_recall_curve, average_precision_score,
)

# ── Paths ────────────────────────────────────────────────────────────────
DATA_DIR   = Path(r'C:\Users\Admin\Documents\nlp\harnn_project\data\process_data')
OUTPUT_DIR = Path(r'C:\Users\Admin\Documents\nlp\harnn_project\output')
CKPT_PATH  = OUTPUT_DIR / 'models' / 'checkpoints' / 'best_model.pt'
FIG_DIR    = OUTPUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEED       = 42
MAX_LEN    = 512
BATCH_SIZE = 64
EMBED_DIM  = 100
HIDDEN     = 256
THRESHOLD  = 0.5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Load data ────────────────────────────────────────────────────────────
with open(DATA_DIR / 'dataset.json',   encoding='utf-8') as f: articles  = json.load(f)
with open(DATA_DIR / 'vocab.json',     encoding='utf-8') as f: vocab     = json.load(f)
with open(DATA_DIR / 'label_map.json', encoding='utf-8') as f: label_map = json.load(f)

VOCAB_SIZE  = len(vocab)
NUM_CLASSES = [len(label_map['l1']), len(label_map['l2']), len(label_map['l3'])]

# idx → tên nhãn
IDX2LABEL = {lv: {v: k for k, v in label_map[lv].items()} for lv in ['l1','l2','l3']}

# ── Dataset ───────────────────────────────────────────────────────────────
class VnExpressDataset(Dataset):
    def __init__(self, articles, vocab, max_len=MAX_LEN):
        self.articles, self.vocab, self.max_len = articles, vocab, max_len
    def __len__(self): return len(self.articles)
    def __getitem__(self, idx):
        a   = self.articles[idx]
        ids = [self.vocab.get(t, 1) for t in a['tokens']][:self.max_len]
        ids += [0] * (self.max_len - len(ids))
        return (
            torch.tensor(ids,         dtype=torch.long),
            torch.tensor(a['vec_l1'], dtype=torch.float),
            torch.tensor(a['vec_l2'], dtype=torch.float),
            torch.tensor(a['vec_l3'], dtype=torch.float),
        )

dataset = VnExpressDataset(articles, vocab)
n       = len(dataset)
n_train = int(n * 0.8)
n_val   = int(n * 0.1)
n_test  = n - n_train - n_val

gen = torch.Generator().manual_seed(SEED)
_, _, test_set = random_split(dataset, [n_train, n_val, n_test], generator=gen)
test_loader    = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'Test set: {len(test_set)} bài')

# ── Model ─────────────────────────────────────────────────────────────────
class HARNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes_per_level, dropout=0.5):
        super().__init__()
        self.num_levels  = len(num_classes_per_level)
        self.hidden_size = hidden_size
        self.embedding   = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.bigru       = nn.GRU(embed_dim, hidden_size, bidirectional=True, batch_first=True)
        self.dropout     = nn.Dropout(dropout)
        self.attention   = nn.ModuleList([nn.Linear(hidden_size*2, 1) for _ in range(self.num_levels)])
        self.ham         = nn.LSTMCell(hidden_size*2, hidden_size)
        self.classifiers = nn.ModuleList([nn.Linear(hidden_size*3, n) for n in num_classes_per_level])
    def forward(self, x):
        B      = x.size(0)
        doc, _ = self.bigru(self.dropout(self.embedding(x)))
        doc    = self.dropout(doc)
        h = torch.zeros(B, self.hidden_size, device=x.device)
        c = torch.zeros(B, self.hidden_size, device=x.device)
        preds = []
        for lv in range(self.num_levels):
            context = (torch.softmax(self.attention[lv](doc), dim=1) * doc).sum(dim=1)
            h, c    = self.ham(context, (h, c))
            preds.append(torch.sigmoid(self.classifiers[lv](self.dropout(torch.cat([context, h], dim=-1)))))
        return preds

model = HARNN(VOCAB_SIZE, EMBED_DIM, HIDDEN, NUM_CLASSES).to(device)
ckpt  = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Checkpoint: epoch {ckpt['epoch']}  val F1-L1={ckpt['val_f1_l1']:.3f}")

# ── Collect predictions ───────────────────────────────────────────────────
@torch.no_grad()
def get_preds_labels():
    P = [[] for _ in range(3)]
    L = [[] for _ in range(3)]
    for ids, l1, l2, l3 in test_loader:
        out = model(ids.to(device))
        for i, lbl in enumerate([l1, l2, l3]):
            P[i].append(out[i].cpu().numpy())
            L[i].append(lbl.numpy())
    return ([np.concatenate(p) for p in P],
            [np.concatenate(l) for l in L])

probs, labels = get_preds_labels()
preds = [(p >= THRESHOLD).astype(int) for p in probs]
print(f'Predictions collected: {len(probs[0])} bài')


## Cell 2 — Confusion Matrix L1
Thấy domain nào hay bị nhầm sang domain nào.


In [ ]:
# Lấy nhãn đơn (argmax) cho confusion matrix
true_l1 = np.argmax(labels[0], axis=1)
pred_l1 = np.argmax(probs[0],  axis=1)

label_names = [IDX2LABEL['l1'][i] for i in range(NUM_CLASSES[0])]
cm = confusion_matrix(true_l1, pred_l1)

fig, ax = plt.subplots(figsize=(11, 9))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(ax=ax, colorbar=True, cmap='Blues', xticks_rotation=45)

ax.set_title('Confusion Matrix — L1 (domain)', fontsize=14, pad=15)
ax.set_xlabel('Dự đoán',  fontsize=11)
ax.set_ylabel('Thực tế', fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / 'confusion_matrix_l1.png', dpi=150)
plt.show()

# In các cặp nhầm lẫn nhiều nhất
print('\nTop nhầm lẫn L1:')
errors = [(cm[i,j], label_names[i], label_names[j])
          for i in range(len(label_names))
          for j in range(len(label_names)) if i != j and cm[i,j] > 0]
for cnt, true, pred in sorted(errors, reverse=True)[:8]:
    print(f'  {true:<15} → {pred:<15}  {cnt} bài')


## Cell 3 — F1 theo từng nhãn L2
Tìm nhãn nào model dự đoán tốt, nhãn nào kém.


In [ ]:
# F1 per-label cho L2
valid_mask = labels[1].sum(axis=1) > 0   # bài có nhãn L2
f1_per_label = f1_score(
    labels[1][valid_mask],
    preds[1][valid_mask],
    average=None,
    zero_division=0,
)

label_names_l2 = [IDX2LABEL['l2'][i] for i in range(NUM_CLASSES[1])]
support        = labels[1][valid_mask].sum(axis=0).astype(int)

# Sắp xếp theo F1 giảm dần
order      = np.argsort(f1_per_label)[::-1]
names_sort = [label_names_l2[i] for i in order]
f1_sort    = f1_per_label[order]
sup_sort   = support[order]

fig, ax = plt.subplots(figsize=(10, max(6, len(names_sort) * 0.38)))
bars = ax.barh(range(len(names_sort)), f1_sort, color=[
    '#2ecc71' if v >= 0.7 else '#f39c12' if v >= 0.4 else '#e74c3c'
    for v in f1_sort
])

# Thêm số support bên phải
for i, (f1, sup) in enumerate(zip(f1_sort, sup_sort)):
    ax.text(f1 + 0.01, i, f'{f1:.2f}  (n={sup})', va='center', fontsize=8.5)

ax.set_yticks(range(len(names_sort)))
ax.set_yticklabels(names_sort, fontsize=9)
ax.set_xlim(0, 1.15)
ax.axvline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax.set_xlabel('F1-score')
ax.set_title('F1 theo từng nhãn L2', fontsize=13)

# Legend màu
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#2ecc71', label='F1 ≥ 0.7'),
    Patch(color='#f39c12', label='0.4 ≤ F1 < 0.7'),
    Patch(color='#e74c3c', label='F1 < 0.4'),
], fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig(FIG_DIR / 'f1_per_label_l2.png', dpi=150)
plt.show()

# In tóm tắt
good = (f1_per_label >= 0.7).sum()
mid  = ((f1_per_label >= 0.4) & (f1_per_label < 0.7)).sum()
bad  = (f1_per_label < 0.4).sum()
print(f'F1 ≥ 0.7: {good} nhãn  |  0.4–0.7: {mid} nhãn  |  < 0.4: {bad} nhãn')


## Cell 4 — Precision-Recall Curve
Thấy được trade-off khi thay đổi threshold cho từng level.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#3498db', '#e67e22', '#2ecc71']

for i, (lv, ax, color) in enumerate(zip(['L1','L2','L3'], axes, colors)):
    valid = labels[i].sum(axis=1) > 0
    if not valid.any():
        ax.set_title(f'{lv} — không có dữ liệu')
        continue

    # Micro-average PR curve
    p_curve, r_curve, thresholds = precision_recall_curve(
        labels[i][valid].ravel(), probs[i][valid].ravel()
    )
    auprc = average_precision_score(labels[i][valid], probs[i][valid], average='micro')

    ax.plot(r_curve, p_curve, color=color, linewidth=2, label=f'AUPRC={auprc:.3f}')

    # Đánh dấu vị trí threshold=0.5
    idx = np.argmin(np.abs(thresholds - 0.5)) if len(thresholds) > 0 else None
    if idx is not None:
        ax.scatter(r_curve[idx], p_curve[idx], s=80, color='red', zorder=5,
                   label=f'threshold=0.5\nP={p_curve[idx]:.2f} R={r_curve[idx]:.2f}')

    ax.set_xlim(0, 1);  ax.set_ylim(0, 1.05)
    ax.set_xlabel('Recall');  ax.set_ylabel('Precision')
    ax.set_title(f'PR Curve — {lv}', fontsize=12)
    ax.legend(fontsize=8.5)
    ax.grid(alpha=0.3)

plt.suptitle('Precision-Recall Curve theo level', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'pr_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print('Đọc biểu đồ:')
print('  Điểm đỏ = threshold 0.5 hiện tại')
print('  Muốn Recall cao hơn → kéo threshold xuống (điểm đỏ dịch phải)')
print('  Muốn Precision cao hơn → kéo threshold lên (điểm đỏ dịch trái)')


## Cell 5 — Phân phối xác suất dự đoán
Kiểm tra model có tự tin không, hay toàn ra xác suất lưng chừng 0.4–0.6.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, (lv, ax) in enumerate(zip(['L1','L2','L3'], axes)):
    # Max prob của mỗi bài (nhãn được chọn)
    max_probs = probs[i].max(axis=1)

    ax.hist(max_probs, bins=30, color='#3498db', edgecolor='white', alpha=0.85)
    ax.axvline(THRESHOLD, color='red', linestyle='--', linewidth=1.5,
               label=f'threshold={THRESHOLD}')

    mean_p = max_probs.mean()
    ax.axvline(mean_p, color='orange', linestyle='--', linewidth=1.5,
               label=f'mean={mean_p:.2f}')

    confident = (max_probs >= 0.7).sum()
    uncertain = (max_probs < 0.5).sum()

    ax.set_xlabel('Max probability')
    ax.set_ylabel('Số bài')
    ax.set_title(f'Phân phối xác suất — {lv}', fontsize=11)
    ax.legend(fontsize=8.5)
    ax.text(0.05, 0.92, f'Tự tin (≥0.7): {confident}\nBăn khoăn (<0.5): {uncertain}',
            transform=ax.transAxes, fontsize=8.5,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('Phân phối xác suất dự đoán (max prob per sample)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'prob_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('Đọc biểu đồ:')
print('  Phần lớn bài nằm gần 1.0 → model tự tin → tốt')
print('  Phần lớn bài nằm 0.4–0.6 → model chưa tự tin → thử giảm dropout hoặc train thêm')
